# GP-DDM Visualization Standardized

本 Notebook 统一整理：
1. v1 / v2.4.x / v2.5 / real data 版本对比
2. P/T/W → SPE 的 GP 均值/方差热图与 3D 响应面
3. P/T/W → DDM 参数（v/a/t0/z）的 GP 映射
4. Sigmoid vs GP 对比（突出 GP 的非线性与不确定性优势）

输出图默认保存到：`3_Figures/GP_DDM_Visualization_Standardized/` 下的主题+版本目录。

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.special import expit
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, RBF, WhiteKernel
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

CURRENT_DIR = Path.cwd()
BASE_DIR = CURRENT_DIR.parent.parent
GEN_DIR = BASE_DIR / '2_Data' / 'Generate_Data'
REAL_DIR = BASE_DIR / '2_Data' / 'Real_Data'
FIG_BASE = BASE_DIR / '3_Figures' / 'GP_DDM_Visualization_Standardized'

FIG_VERSION = FIG_BASE / '01_Version_Compare'
FIG_SPE = FIG_BASE / '02_GP_SPE_Surface'
FIG_GPMAP = FIG_BASE / '03_GP_DDM_Mapping'
FIG_SIG_GGP = FIG_BASE / '04_Sigmoid_vs_GP'
FIG_BOUND = FIG_BASE / '05_GP_Boundary'
for d in [FIG_VERSION, FIG_SPE, FIG_GPMAP, FIG_SIG_GGP, FIG_BOUND]:
    d.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 200)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.unicode_minus'] = False

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    raise FileNotFoundError(f'No file found among: {paths}')

def to_num(s):
    return pd.to_numeric(s, errors='coerce')

def to_ms(rt):
    x = to_num(rt)
    if np.nanmax(x.to_numpy(dtype=float)) <= 10:
        return x * 1000.0
    return x

def save_fig(fig, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=220, bbox_inches='tight')
    plt.close(fig)

print('BASE_DIR =', BASE_DIR)
print('GEN_DIR  =', GEN_DIR)
print('REAL_DIR =', REAL_DIR)


BASE_DIR = d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design
GEN_DIR  = d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Generate_Data
REAL_DIR = d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\2_Data\Real_Data


In [3]:
# =========================
# 1) 载入并标准化数据
# =========================
v1_path = first_existing(sorted(GEN_DIR.glob('gp_ddm_simulation_v1*.csv')))
v24_path = first_existing([
    GEN_DIR / 'Generate_Data_v2.4.5_checks' / 'gp_ddm_v2.4.5_large.csv',
    GEN_DIR / 'Generate_Data_v2.4.4_checks' / 'gp_ddm_v2.4.4_large.csv',
    GEN_DIR / 'Generate_Data_v2.4.3_checks' / 'gp_ddm_v2.4.3_large.csv',
    GEN_DIR / 'Generate_Data_v2.4_checks' / 'gp_ddm_v2.4_large.csv',
])
v25_path = first_existing([
    GEN_DIR / 'Generate_Data_v2.5' / 'gp_ddm_v2.5_2000.csv',
])
real_path = REAL_DIR / 'EXP_data_combined.csv'

def standardize_generated(df, version_name):
    df = df.copy()
    colmap = {c.lower(): c for c in df.columns}
    get = lambda name: df[colmap[name]].copy() if name in colmap else pd.Series([np.nan] * len(df))
    out = pd.DataFrame({
        'version': version_name,
        'source_type': 'generated',
        'subject': to_num(get('subject')) if 'subject' in colmap else to_num(get('subjectid')),
        'trial': to_num(get('trial')) if 'trial' in colmap else to_num(get('trialid')),
        'P': to_num(get('p')),
        'T': to_num(get('t')),
        'W': to_num(get('w')),
        'M': to_num(get('m')),
        'label': get('label').astype(str).str.lower(),
        'v': to_num(get('v')),
        'a': to_num(get('a')),
        't0': to_num(get('t0')),
        'z': to_num(get('z')),
        'RT_ms': to_ms(get('rt')),
        'response': to_num(get('response')),
        'correct': to_num(get('correct')) if 'correct' in colmap else to_num(get('response')),
        'is_valid': get('is_valid') if 'is_valid' in colmap else True,
        'is_timeout': get('is_timeout') if 'is_timeout' in colmap else False,
    })
    return out

def standardize_real(df):
    df = df.copy()
    out = pd.DataFrame({
        'version': 'real',
        'source_type': 'real',
        'subject': to_num(df['subjectID'] if 'subjectID' in df.columns else df['subject']),
        'trial': to_num(df['trialID'] if 'trialID' in df.columns else df['trial']),
        'P': to_num(df['P']),
        'T': to_num(df['T']),
        'W': to_num(df['W']),
        'M': to_num(df['P']) + to_num(df['T']) + to_num(df['W']),
        'label': df['Label'].astype(str).str.lower(),
        'v': np.nan,
        'a': np.nan,
        't0': np.nan,
        'z': np.nan,
        'RT_ms': to_ms(df['RT']),
        'response': df['Response'].astype(str),
        'correct': to_num(df['Correct']),
        'is_valid': to_num(df['Correct']).notna(),
        'is_timeout': to_num(df['Correct']).isna(),
    })
    return out

dfs = {
    'v1': standardize_generated(pd.read_csv(v1_path), 'v1'),
    'v2.4.x': standardize_generated(pd.read_csv(v24_path), 'v2.4.x'),
    'v2.5': standardize_generated(pd.read_csv(v25_path), 'v2.5'),
    'real': standardize_real(pd.read_csv(real_path)),
}

for name, df in dfs.items():
    print(name, df.shape, '| labels =', sorted(df['label'].dropna().unique())[:5])

def build_design_spe(df):
    tmp = df.dropna(subset=['P', 'T', 'W', 'label', 'RT_ms']).copy()
    g = tmp.groupby(['P', 'T', 'W', 'label'], as_index=False)['RT_ms'].mean()
    pivot = g.pivot_table(index=['P', 'T', 'W'], columns='label', values='RT_ms')
    if 'self' not in pivot.columns or 'stranger' not in pivot.columns:
        return pd.DataFrame()
    pivot = pivot.dropna(subset=['self', 'stranger']).copy()
    pivot = pivot.reset_index()
    pivot['SPE_ms'] = pivot['self'] - pivot['stranger']
    return pivot.rename(columns={'self': 'RT_self_ms', 'stranger': 'RT_stranger_ms'})

design_tables = {name: build_design_spe(df) for name, df in dfs.items()}
for name, tbl in design_tables.items():
    print(name, 'design table:', tbl.shape)


v1 (243, 18) | labels = ['self', 'stranger']
v2.4.x (120000, 18) | labels = ['self', 'stranger']
v2.5 (120000, 18) | labels = ['self', 'stranger']
real (26616, 18) | labels = ['self', 'stranger']
v1 design table: (5, 6)
v2.4.x design table: (1919, 6)
v2.5 design table: (2000, 6)
real design table: (6, 6)


In [4]:
# =========================
# 2) 版本对比链路
# =========================
summary_rows = []
for name, df in dfs.items():
    spe_tbl = design_tables[name]
    row = {
        'version': name,
        'n_rows': len(df),
        'n_design_points': len(spe_tbl),
        'mean_rt_ms': df['RT_ms'].mean(),
        'self_rt_ms': df.loc[df['label'] == 'self', 'RT_ms'].mean(),
        'stranger_rt_ms': df.loc[df['label'] == 'stranger', 'RT_ms'].mean(),
        'spe_ms': spe_tbl['SPE_ms'].mean() if len(spe_tbl) else np.nan,
        'spe_sd': spe_tbl['SPE_ms'].std() if len(spe_tbl) else np.nan,
        'accuracy_rate': df['correct'].mean() if 'correct' in df.columns else np.nan,
        'valid_rate': pd.Series(df['is_valid']).astype(float).mean() if 'is_valid' in df.columns else np.nan,
    }
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(FIG_VERSION / 'version_summary.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
summary.plot(kind='bar', x='version', y='spe_ms', ax=axes[0], color=['#5470C6', '#91CC75', '#FAC858', '#EE6666'])
axes[0].set_title('SPE by Version')
axes[0].set_ylabel('SPE (ms)')
axes[0].grid(alpha=0.25)
summary.plot(kind='bar', x='version', y='mean_rt_ms', ax=axes[1], color='#73C0DE')
axes[1].set_title('Mean RT by Version')
axes[1].set_ylabel('RT (ms)')
axes[1].grid(alpha=0.25)
summary.plot(kind='bar', x='version', y='n_design_points', ax=axes[2], color='#FC8452')
axes[2].set_title('Design Points by Version')
axes[2].set_ylabel('Count')
axes[2].grid(alpha=0.25)
plt.tight_layout()
save_fig(fig, FIG_VERSION / 'version_compare_overview.png')
print('Version compare saved to', FIG_VERSION)


,version,n_rows,n_design_points,mean_rt_ms,self_rt_ms,stranger_rt_ms,spe_ms,spe_sd,accuracy_rate,valid_rate
0,v1,243,5,701.132582,550.159852,913.391272,-313.514702,285.404820,0.987654,1.000000
1,v2.4.x,120000,1919,562.900275,521.520054,626.240843,-72.176618,90.545047,0.501242,1.000000
2,v2.5,120000,2000,721.735467,522.367392,921.103542,-398.736150,228.036249,0.770375,0.735442
3,real,26616,6,562.967983,550.568771,575.744865,-19.270200,37.735062,0.429742,1.000000


Version compare saved to d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\GP_DDM_Visualization_Standardized\01_Version_Compare


In [5]:
# =========================
# 3) GP / Sigmoid 共用函数
# =========================
def fit_gp(df, target, feature_cols=('P', 'T', 'W'), max_points=2500, random_state=42):
    clean = df.dropna(subset=list(feature_cols) + [target]).copy()
    # 先按设计点聚合，避免同一 P/T/W 被试/试次重复导致 GP 训练点爆炸
    clean = clean.groupby(list(feature_cols), as_index=False)[target].mean()
    # 如果仍然过大，再做等比例抽样，保证 GP 可运行
    if len(clean) > max_points:
        clean = clean.sample(n=max_points, random_state=random_state).sort_values(list(feature_cols)).reset_index(drop=True)
    X = clean.loc[:, feature_cols].astype(float).to_numpy()
    y = clean[target].astype(float).to_numpy()
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    kernel = C(1.0, (0.1, 10.0)) * RBF(length_scale=np.ones(Xs.shape[1]), length_scale_bounds=(0.1, 50.0)) + WhiteKernel(noise_level=1e-4, noise_level_bounds=(1e-6, 1.0))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=random_state)
    gp.fit(Xs, y)
    return gp, scaler, clean

def make_grid(df, x_col='P', y_col='T', fixed_col='W', fixed_value=None, n=50):
    x = np.linspace(df[x_col].quantile(0.05), df[x_col].quantile(0.95), n)
    y = np.linspace(df[y_col].quantile(0.05), df[y_col].quantile(0.95), n)
    Xg, Yg = np.meshgrid(x, y)
    if fixed_value is None:
        fixed_value = df[fixed_col].median()
    grid = pd.DataFrame({x_col: Xg.ravel(), y_col: Yg.ravel(), fixed_col: fixed_value})
    return Xg, Yg, grid, fixed_value

def predict_surface(gp, scaler, grid_df, feature_cols=('P', 'T', 'W')):
    Xg = scaler.transform(grid_df.loc[:, feature_cols].astype(float).to_numpy())
    mean, std = gp.predict(Xg, return_std=True)
    return mean, std

def plot_heatmap_and_boundary(df, target, version_name, x_col='P', y_col='T', fixed_col='W', fixed_value=None, out_dir=FIG_SPE):
    gp, scaler, clean = fit_gp(df, target)
    Xg, Yg, grid, fixed_value = make_grid(clean, x_col=x_col, y_col=y_col, fixed_col=fixed_col, fixed_value=fixed_value)
    mean, std = predict_surface(gp, scaler, grid)
    mean = mean.reshape(Xg.shape)
    std = std.reshape(Xg.shape)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    im0 = axes[0].contourf(Xg, Yg, mean, levels=20, cmap='viridis')
    axes[0].scatter(clean[x_col], clean[y_col], s=12, c='white', alpha=0.35, edgecolor='none')
    axes[0].set_title(f'{version_name} | GP mean: {target} ({fixed_col}={fixed_value:.2f})')
    axes[0].set_xlabel(x_col)
    axes[0].set_ylabel(y_col)
    fig.colorbar(im0, ax=axes[0], label=f'Predicted {target}')

    im1 = axes[1].contourf(Xg, Yg, std, levels=20, cmap='magma')
    cutoff = np.quantile(std, 0.80)
    axes[1].contour(Xg, Yg, std, levels=[cutoff], colors='cyan', linewidths=2)
    axes[1].scatter(clean[x_col], clean[y_col], s=12, c='white', alpha=0.35, edgecolor='none')
    axes[1].set_title(f'{version_name} | GP std boundary: {target}')
    axes[1].set_xlabel(x_col)
    axes[1].set_ylabel(y_col)
    fig.colorbar(im1, ax=axes[1], label='Prediction std')

    plt.tight_layout()
    out_path = out_dir / version_name / f'{version_name}_{target}_{x_col}_{y_col}_{fixed_col}_gp_mean_std.png'
    save_fig(fig, out_path)
    return gp, scaler, clean, mean, std, out_path, fixed_value

def plot_3d_surface(df, target, version_name, x_col='T', y_col='W', fixed_col='P', fixed_value=None, out_dir=FIG_SPE):
    gp, scaler, clean = fit_gp(df, target)
    Xg, Yg, grid, fixed_value = make_grid(clean, x_col=x_col, y_col=y_col, fixed_col=fixed_col, fixed_value=fixed_value)
    mean, std = predict_surface(gp, scaler, grid)
    Z = mean.reshape(Xg.shape)
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(Xg, Yg, Z, cmap='viridis', alpha=0.88, linewidth=0, antialiased=True)
    sel = np.abs(clean[fixed_col] - fixed_value) <= max(1.0, clean[fixed_col].std() * 0.15)
    ax.scatter(clean.loc[sel, x_col], clean.loc[sel, y_col], clean.loc[sel, target], c='red', s=18, alpha=0.6, label='Observed points')
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_zlabel(target)
    ax.set_title(f'{version_name} | 3D GP surface for {target} ({fixed_col}={fixed_value:.2f})')
    fig.colorbar(surf, shrink=0.65, label=f'Predicted {target}')
    ax.legend(loc='best')
    plt.tight_layout()
    out_path = out_dir / version_name / f'{version_name}_{target}_{x_col}_{y_col}_{fixed_col}_gp_3d.png'
    save_fig(fig, out_path)
    return out_path

def sigmoid_features(X, centers=None, scales=None):
    X = np.asarray(X, dtype=float)
    if centers is None:
        centers = np.median(X, axis=0)
    if scales is None:
        scales = np.std(X, axis=0) + 1e-6
    Z = expit((X - centers) / scales)
    return np.column_stack([np.ones(len(X)), Z, X])

def fit_sigmoid_baseline(X_train, y_train, X_test):
    centers = np.median(X_train, axis=0)
    scales = np.std(X_train, axis=0) + 1e-6
    Phi_train = sigmoid_features(X_train, centers, scales)
    Phi_test = sigmoid_features(X_test, centers, scales)
    model = Ridge(alpha=1.0)
    model.fit(Phi_train, y_train)
    return model.predict(Phi_test), model, centers, scales

def gp_vs_sigmoid_compare(df, target, version_name, x_cols=('P', 'T', 'W'), out_dir=FIG_SIG_GGP):
    clean = df.dropna(subset=list(x_cols) + [target]).copy()
    X = clean.loc[:, x_cols].astype(float).to_numpy()
    y = clean[target].astype(float).to_numpy()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    gp, scaler, _ = fit_gp(pd.DataFrame(clean), target, feature_cols=x_cols)
    gp_pred = gp.predict(scaler.transform(X_test))
    sig_pred, _, _, _ = fit_sigmoid_baseline(X_train, y_train, X_test)

    metrics = pd.DataFrame([
        {'model': 'GP', 'rmse': np.sqrt(mean_squared_error(y_test, gp_pred)), 'r2': r2_score(y_test, gp_pred)},
        {'model': 'Sigmoid', 'rmse': np.sqrt(mean_squared_error(y_test, sig_pred)), 'r2': r2_score(y_test, sig_pred)},
    ])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, pred, title in [
        (axes[0], gp_pred, 'GP'),
        (axes[1], sig_pred, 'Sigmoid baseline'),
    ]:
        ax.scatter(y_test, pred, s=20, alpha=0.75)
        mn = min(y_test.min(), pred.min())
        mx = max(y_test.max(), pred.max())
        ax.plot([mn, mx], [mn, mx], 'k--', lw=1)
        ax.set_title(f'{version_name} | {target} | {title}')
        ax.set_xlabel('Observed')
        ax.set_ylabel('Predicted')
        ax.grid(alpha=0.25)
    plt.suptitle(f'Sigmoid vs GP comparison: {version_name} / {target}')
    plt.tight_layout()
    out_path = out_dir / version_name / f'{version_name}_{target}_sigmoid_vs_gp.png'
    save_fig(fig, out_path)
    metrics.to_csv(out_dir / version_name / f'{version_name}_{target}_sigmoid_vs_gp_metrics.csv', index=False, encoding='utf-8-sig')
    return metrics, out_path


In [6]:
# =========================
# 4) SPE 响应面 + 边界区 + 3D 图
# =========================
for version_name in ['v1', 'v2.4.x', 'v2.5', 'real']:
    df = design_tables[version_name].copy()
    if df.empty:
        continue
    # 2D：P/T 平面，固定 W 为中位数
    plot_heatmap_and_boundary(df, 'SPE_ms', version_name, x_col='P', y_col='T', fixed_col='W', out_dir=FIG_SPE)
    # 3D：T/W 平面，固定 P 为中位数（与现有 GP-SPE-Explore-3D 风格一致）
    plot_3d_surface(df, 'SPE_ms', version_name, x_col='T', y_col='W', fixed_col='P', out_dir=FIG_SPE)

print('SPE surfaces saved to', FIG_SPE)


c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(

SPE surfaces saved to d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\GP_DDM_Visualization_Standardized\02_GP_SPE_Surface


In [7]:
# =========================
# 5) P/T/W → DDM 参数 GP 映射
# =========================
for version_name in ['v2.4.x', 'v2.5']:
    df = dfs[version_name].copy()
    targets = [c for c in ['v', 'a', 't0', 'z'] if c in df.columns and df[c].notna().any()]
    for target in targets:
        # 只对核心参数先画二维/三维，避免一次性训练太多 GP 模型
        plot_heatmap_and_boundary(df, target, version_name, x_col='P', y_col='T', fixed_col='W', out_dir=FIG_GPMAP)
        if target in ['v', 'a']:
            plot_3d_surface(df, target, version_name, x_col='T', y_col='W', fixed_col='P', out_dir=FIG_GPMAP)

print('DDM parameter maps saved to', FIG_GPMAP)


c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.w

DDM parameter maps saved to d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\GP_DDM_Visualization_Standardized\03_GP_DDM_Mapping


In [8]:
# =========================
# 6) Sigmoid vs GP 对比（突出 GP 特色）
# =========================
compare_rows = []
for version_name in ['v1', 'v2.4.x', 'v2.5', 'real']:
    df = design_tables[version_name].copy()
    if df.empty:
        continue
    metrics, out_path = gp_vs_sigmoid_compare(df, 'SPE_ms', version_name, out_dir=FIG_SIG_GGP)
    metrics['version'] = version_name
    compare_rows.append(metrics)

compare_df = pd.concat(compare_rows, ignore_index=True) if compare_rows else pd.DataFrame()
display(compare_df)
if not compare_df.empty:
    compare_df.to_csv(FIG_SIG_GGP / 'sigmoid_vs_gp_all_versions.csv', index=False, encoding='utf-8-sig')
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    pivot = compare_df.pivot(index='version', columns='model', values='rmse')
    pivot.plot(kind='bar', ax=ax)
    ax.set_title('Sigmoid vs GP RMSE on SPE')
    ax.set_ylabel('RMSE')
    ax.grid(alpha=0.25)
    plt.tight_layout()
    save_fig(fig, FIG_SIG_GGP / 'sigmoid_vs_gp_rmse_overview.png')

print('Sigmoid vs GP outputs saved to', FIG_SIG_GGP)


c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 50.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\蔡振辛\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


,model,rmse,r2,version
0,GP,0.000284,1.000000,v1
1,Sigmoid,375.598858,-3.850728,v1
2,GP,55.851405,0.642608,v2.4.x
3,Sigmoid,61.397561,0.568104,v2.4.x
4,GP,70.006589,0.904864,v2.5
5,Sigmoid,97.582966,0.815153,v2.5
6,GP,0.000195,1.000000,real
7,Sigmoid,15.205149,-41.665939,real


Sigmoid vs GP outputs saved to d:\GitHub_programe\GitHub\Guassion-Process-Experiment-Design\3_Figures\GP_DDM_Visualization_Standardized\04_Sigmoid_vs_GP
